# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [66]:
# Fill in your information
name = "Abdur Rahman"           # Write your full name here
email = "abdurrahmansoftw@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Abdur Rahman
Email : abdurrahmansoftw@gmail.com


#### Imports package

In [67]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from warnings import filterwarnings
filterwarnings('ignore')

---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [68]:
# Question 1

df = pd.read_csv('/content/house_price_regression_dataset.csv')

X = df.drop('House_Price', axis=1)
y = df['House_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms'])
])

In [69]:
df.head(10)

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
5,3944,5,3,1990,2.475930,2,8,8.797970e+05
6,3671,1,2,2012,4.911960,0,1,8.144279e+05
7,3419,1,1,1972,2.805281,1,1,7.034131e+05
8,630,3,3,1997,1.014286,1,8,1.738750e+05
9,2185,4,2,1981,3.941604,2,5,5.041765e+05


In [70]:
df.sample(5, random_state=42)

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
521,4012,3,1,2016,2.098092,1,5,9.010005e+05
737,2310,3,1,1988,1.369622,1,4,4.945375e+05
740,4708,1,3,1962,1.792970,1,8,9.494042e+05
660,4932,2,1,1972,4.479598,1,2,1.040389e+06
411,3646,1,1,1994,3.980987,0,9,7.940100e+05


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [71]:
num_imputer = SimpleImputer(strategy='mean')

X_train = pd.DataFrame(num_imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(num_imputer.transform(X_test), columns=X_test.columns)

print(X_train.shape)
print(X_test.shape)

(800, 7)
(200, 7)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [72]:
#Question 3

x = X_train['Square_Footage'].values
y_t = y_train.values
x = (x - x.mean()) / x.std()

m, c = 0, 0

for _ in range(1000):
    e = (m * x + c) - y_t
    m -= 0.01 * (1/len(x)) * np.dot(e, x)
    c -= 0.01 * (1/len(x)) * np.sum(e)

print(f'm = {m:.4f}')
print(f'c = {c:.4f}')

m = 251083.6878
c = 618549.3497


---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [73]:
#Question 4
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms'])
])

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Square_Footage', 'Num_Bedrooms',
                                  'Num_Bathrooms'])])


## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [74]:
# Question 5
pipeline = Pipeline([
    ('pre', preprocessor),
    ('model', SGDRegressor(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')
print(f'R2:   {r2_score(y_test, y_pred):.4f}')
print(pd.DataFrame({'Actual': y_test.values[:10], 'Predicted': y_pred[:10]}))

RMSE: 29097.9447
R2:   0.9869
         Actual     Predicted
0  9.010005e+05  8.509483e+05
1  4.945375e+05  5.085160e+05
2  9.494042e+05  9.895024e+05
3  1.040389e+06  1.025982e+06
4  7.940100e+05  7.571821e+05
5  7.240336e+05  7.645012e+05
6  9.984392e+05  1.005613e+06
7  9.097134e+05  9.035409e+05
8  7.926815e+05  7.944079e+05
9  9.474908e+05  8.873596e+05


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [75]:
# Question 6
pipeline = Pipeline([
    ('pre', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')
print(f'R2:   {r2_score(y_test, y_pred):.4f}')
print(pd.DataFrame({'Actual': y_test.values[:10], 'Predicted': y_pred[:10]}))

RMSE: 29092.5684
R2:   0.9869
         Actual     Predicted
0  9.010005e+05  8.508310e+05
1  4.945375e+05  5.084529e+05
2  9.494042e+05  9.893012e+05
3  1.040389e+06  1.025789e+06
4  7.940100e+05  7.569829e+05
5  7.240336e+05  7.643666e+05
6  9.984392e+05  1.005554e+06
7  9.097134e+05  9.035295e+05
8  7.926815e+05  7.942511e+05
9  9.474908e+05  8.871881e+05


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [76]:
# Question 7
sample = df.sample(100, random_state=42)

X_s = np.column_stack([np.ones(100), sample[['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']].values])
y_s = sample['House_Price'].values

theta = np.linalg.inv(X_s.T @ X_s) @ X_s.T @ y_s

print(f'Intercept:      {theta[0]:.4f}')
print(f'Square_Footage: {theta[1]:.4f}')
print(f'Num_Bedrooms:   {theta[2]:.4f}')
print(f'Num_Bathrooms:  {theta[3]:.4f}')

Intercept:      20888.2595
Square_Footage: 202.3082
Num_Bedrooms:   8303.2810
Num_Bathrooms:  3020.3285
